In [ ]:
# import geemap
import ee

# ee.Authenticate()
ee.Initialize()

# Leafmap for bbox

In [ ]:
import leafmap

In [ ]:
m = leafmap.Map(center=[16.983124, 95.449798], zoom=15)
m.add_basemap("SATELLITE")
m

In [ ]:
if m.user_roi_bounds() is not None:
    bbox = m.user_roi_bounds()
else:
    bbox = [95.508, 16.9859, 95.513, 16.99]

In [ ]:
bbox

# ESRI

In [ ]:
# ## ESRI Source
# from getData import tms_to_geotiff
# from PIL import Image
# import numpy as np

# # # Single image 
# # from getData import tms_to_geotiff
# # from PIL import Image
# # import numpy as np

# # nr = 1
# # image = "D:/SIG/Pantanaw/Test_image/test_source_image/esri_imagery_test01.tif" 
# # satImg = tms_to_geotiff(output=image, bbox=bbox, zoom=19, source="ESRI", overwrite=True, return_image=True)

# zoom_levels = range(15, 20)
# output_dir = "D:/SIG/Pantanaw/Test_image/test_source_image/"
# base_filename = "esri_imagery_test"
# for zoom in zoom_levels:
#     output_file = f"{output_dir}{base_filename}_z{zoom}.tif"
#     try:
#         print(f"Downloading zoom level {zoom} to {output_file}...")
#         satImg = tms_to_geotiff(
#             output=output_file, 
#             bbox=bbox, 
#             zoom=zoom, 
#             source="ESRI", 
#             overwrite=True, 
#             return_image=True
#         )
#         print(f"Zoom level {zoom} downloaded successfully.")
#     except Exception as e:
#         print(f"Failed to download zoom level {zoom}: {e}")

# BING

In [ ]:
bbox

In [ ]:
from getData import tms_to_geotiff
import numpy as np
import math
import os

import rasterio
from rasterio.merge import merge
from rasterio.transform import from_bounds

def tile_to_lon_lat(x, y, zoom):
    """Convert tile (x, y) at a specified zoom level to longitude and latitude."""
    n = 2 ** zoom
    lon_deg = x / n * 360.0 - 180.0
    lat_rad = math.atan(math.sinh(math.pi * (1 - 2 * y / n)))
    lat_deg = math.degrees(lat_rad)
    return lon_deg, lat_deg

def lon_lat_to_tile(lon, lat, zoom):
    """Convert latitude and longitude to tile (x, y) at a specified zoom level."""
    n = 2 ** zoom
    x = int((lon + 180.0) / 360.0 * n)
    y = int(
        (1.0 - (math.log(math.tan(math.radians(lat)) + 1 / math.cos(math.radians(lat))) / math.pi)) / 2.0 * n
    )
    return x, y

def tile_to_quadkey(x, y, zoom):
    """Generate the quadkey for a given tile (x, y) at zoom level."""
    quadkey = []
    for i in range(zoom, 0, -1):
        digit = 0
        mask = 1 << (i - 1)
        if (x & mask) != 0:
            digit += 1
        if (y & mask) != 0:
            digit += 2
        quadkey.append(str(digit))
    return ''.join(quadkey)

# Set parameters
bbox = [95.508, 16.9859, 95.513, 16.99]  # [min_lon, min_lat, max_lon, max_lat]
zoom = 19
tiles_dir = "D:/SIG/Pantanaw/Test_image/test_source_image/bing/temp"
base_filename = "bing_tile"

# Create output directory if it doesn't exist
os.makedirs(tiles_dir, exist_ok=True)

# Calculate tile ranges for the bounding box
x_min, y_min = lon_lat_to_tile(bbox[0], bbox[1], zoom)
x_max, y_max = lon_lat_to_tile(bbox[2], bbox[3], zoom)

# Ensure y_min < y_max for looping
if y_min > y_max:
    y_min, y_max = y_max, y_min

# Loop over each tile within the bounding box
for x in range(x_min, x_max + 1):
    for y in range(y_min, y_max + 1):
        # Generate the quadkey for the current tile
        quadkey = tile_to_quadkey(x, y, zoom)
        
        # Calculate the bounding box for the current tile
        lon_min, lat_max = tile_to_lon_lat(x, y, zoom)
        lon_max, lat_min = tile_to_lon_lat(x + 1, y + 1, zoom)
        tile_bbox = [lon_min, lat_min, lon_max, lat_max]
        
        # Define the output filename
        output_file = f"{tiles_dir}{base_filename}_z{zoom}_x{x}_y{y}.tif"
        
        # Create the Bing tile URL
        bing_url = f"https://t0.tiles.virtualearth.net/tiles/a{quadkey}.jpeg?g=685&mkt=en-us&n=z"
        
        # Print debugging info
        print(f"Downloading tile: x={x}, y={y}, zoom={zoom}, quadkey={quadkey}")
        print(f"BBox: {tile_bbox}")
        print(f"Output file: {output_file}")

        # Attempt to download the tile
        try:
            tms_to_geotiff(
                output=output_file,
                bbox=tile_bbox,  # Pass the calculated BBox for the tile
                zoom=zoom,
                source=bing_url,  # Pass the full Bing URL as the source
                overwrite=True,
                return_image=False
            )
            print(f"Tile x={x}, y={y} downloaded successfully.")
        except Exception as e:
            print(f"Failed to download tile x={x}, y={y}: {e}")

print("Download process completed.")

# Directory containing the downloaded tiles
# tiles_dir = "D:/SIG/Pantanaw/Test_image/test_source_image/bing/"

# Output file path for the merged GeoTIFF
output_file = "D:/SIG/Pantanaw/Test_image/test_source_image/bing/merged_bing_image.tif"

# Find all .tif files in the directory
tile_files = [
    os.path.join(tiles_dir, f)
    for f in os.listdir(tiles_dir)
    if f.endswith(".tif")
]

# Check if any tiles are found
if not tile_files:
    print("No tile files found. Please make sure tiles are downloaded.")
    exit()

# Open all tile files with rasterio
sources = [rasterio.open(tile) for tile in tile_files]

# Merge tiles
print("Merging tiles...")
mosaic, out_transform = merge(sources)

# Get metadata from the first tile
out_meta = sources[0].meta.copy()

# Update metadata for the merged dataset
out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_transform,
    "crs": sources[0].crs
})

# Write the merged GeoTIFF
print(f"Writing merged GeoTIFF to {output_file}...")
with rasterio.open(output_file, "w", **out_meta) as dest:
    dest.write(mosaic)

print("Merging completed!")

# Close all opened tile files
for src in sources:
    src.close()

# Auto1 Download from GEE Asset

In [ ]:
# import os
# from getData import tms_to_geotiff
# from PIL import Image
# import numpy as np
# areas_list = [
#     "0_vietnam_areas_shp", "10_cambodia_areas_shp", "11_cambodia_areas_shp", "12_cambodia_areas_shp",
#     "13_cambodia_areas_shp", "14_cambodia_areas_shp", "15_cambodia_areas_shp", "16_cambodia_areas_shp",
#     "17_cambodia_areas_shp", "18_cambodia_areas_shp", "19_cambodia_areas_shp", "1_vietnam_areas_shp",
#     "20_cambodia_areas_shp", "21_cambodia_areas_shp", "22_vietnam_areas_shp", "23_vietnam_areas_shp",
#     "24_vietnam_areas_shp", "25_vietnam_areas_shp", "26_vietnam_areas_shp", "27_vietnam_areas_shp",
#     "28_vietnam_areas_shp", "29_vietnam_areas_shp", "2_cambodia_areas_shp", "30_vietnam_areas_shp",
#     "31_vietnam_areas_shp", "32_vietnam_areas_shp", "33_cambodia_areas_shp", "34_cambodia_areas_shp",
#     "35_cambodia_areas_shp", "36_cambodia_areas_shp", "37_cambodia_areas_shp", "38_cambodia_areas_shp",
#     "39_cambodia_areas_shp", "3_cambodia_areas_shp", "40_vietnam_areas_shp", "41_vietnam_areas_shp",
#     "42_vietnam_areas_shp", "43_vietnam_areas_shp", "44_vietnam_areas_shp", "45_vietnam_areas_shp",
#     "46_vietnam_areas_shp", "47_vietnam_areas_shp", "48_vietnam_areas_shp", "49_vietnam_areas_shp",
#     "4_cambodia_areas_shp", "50_vietnam_areas_shp", "51_vietnam_areas_shp", "52_vietnam_areas_shp",
#     "53_vietnam_areas_shp", "54_vietnam_areas_shp", "55_vietnam_areas_shp", "56_vietnam_areas_shp",
#     "57_cambodia_areas_shp", "58_cambodia_areas_shp", "59_cambodia_areas_shp", "5_cambodia_areas_shp",
#     "60_cambodia_areas_shp", "61_cambodia_areas_shp", "6_cambodia_areas_shp", "7_cambodia_areas_shp",
#     "8_cambodia_areas_shp", "9_cambodia_areas_shp"
# ]
# for name in areas_list:
    
#     cambodia_areas = ee.FeatureCollection(f'projects/earthengine-legacy/assets/projects/servir-mekong/khProject/data/{name}')
    
    
#     bounds_info = cambodia_areas.geometry().bounds().getInfo()

    
#     coordinates = bounds_info['coordinates'][0]

    
#     longitudes = [point[0] for point in coordinates]
#     latitudes = [point[1] for point in coordinates]

#     xmin = min(longitudes)
#     xmax = max(longitudes)
#     ymin = min(latitudes)
#     ymax = max(latitudes)

#     print(f"Processing {name}...")
#     print(f"xmin: {xmin}, xmax: {xmax}, ymin: {ymin}, ymax: {ymax}")

    
#     bbox = [xmin, ymin, xmax, ymax]
#     output_image = f"D:/SIG/Sam2_soucre/Crops/imagesZ19/{name}_z19.tif"

#     # Check if the image already exists before downloading
#     if os.path.exists(output_image):
#         print(f"Image for {name} already exists. Skipping download.")
#     else:
#         # Download the image only if it doesn't already exist
#         satImg = tms_to_geotiff(output=output_image, bbox=bbox, zoom=19, source="Satellite", overwrite=True, return_image=True)
#         print(f"Image for {name} saved at {output_image}")

# print("All areas processed.")

In [ ]:
# import csv
# import os
# from getData import tms_to_geotiff
# import ee

# # Authenticate and initialize the Earth Engine API
# ee.Initialize()

# areas_list = [
#     "0_vietnam_areas_shp", "10_cambodia_areas_shp", "11_cambodia_areas_shp", "12_cambodia_areas_shp",
#     "13_cambodia_areas_shp", "14_cambodia_areas_shp", "15_cambodia_areas_shp", "16_cambodia_areas_shp",
#     "17_cambodia_areas_shp", "18_cambodia_areas_shp", "19_cambodia_areas_shp", "1_vietnam_areas_shp",
#     "20_cambodia_areas_shp", "21_cambodia_areas_shp", "22_vietnam_areas_shp", "23_vietnam_areas_shp",
#     "24_vietnam_areas_shp", "25_vietnam_areas_shp", "26_vietnam_areas_shp", "27_vietnam_areas_shp",
#     "28_vietnam_areas_shp", "29_vietnam_areas_shp", "2_cambodia_areas_shp", "30_vietnam_areas_shp",
#     "31_vietnam_areas_shp", "32_vietnam_areas_shp", "33_cambodia_areas_shp", "34_cambodia_areas_shp",
#     "35_cambodia_areas_shp", "36_cambodia_areas_shp", "37_cambodia_areas_shp", "38_cambodia_areas_shp",
#     "39_cambodia_areas_shp", "3_cambodia_areas_shp", "40_vietnam_areas_shp", "41_vietnam_areas_shp",
#     "42_vietnam_areas_shp", "43_vietnam_areas_shp", "44_vietnam_areas_shp", "45_vietnam_areas_shp",
#     "46_vietnam_areas_shp", "47_vietnam_areas_shp", "48_vietnam_areas_shp", "49_vietnam_areas_shp",
#     "4_cambodia_areas_shp", "50_vietnam_areas_shp", "51_vietnam_areas_shp", "52_vietnam_areas_shp",
#     "53_vietnam_areas_shp", "54_vietnam_areas_shp", "55_vietnam_areas_shp", "56_vietnam_areas_shp",
#     "57_cambodia_areas_shp", "58_cambodia_areas_shp", "59_cambodia_areas_shp", "5_cambodia_areas_shp",
#     "60_cambodia_areas_shp", "61_cambodia_areas_shp", "6_cambodia_areas_shp", "7_cambodia_areas_shp",
#     "8_cambodia_areas_shp", "9_cambodia_areas_shp"
# ]

# output_csv = "D:/SIG/Sam2_soucre/Crops/bounding_boxes.csv"
# with open(output_csv, mode='w', newline='') as file:
#     writer = csv.writer(file)
#     # Write headers for CSV file
#     writer.writerow(["Area", "xmin", "xmax", "ymin", "ymax"])

#     for name in areas_list:
#         # Fetch feature collection
#         cambodia_areas = ee.FeatureCollection(f'projects/earthengine-legacy/assets/projects/servir-mekong/khProject/data/{name}')
#         bounds_info = cambodia_areas.geometry().bounds().getInfo()
#         coordinates = bounds_info['coordinates'][0]

#         # Calculate bounding box coordinates 
#         ## Important Part
#         longitudes = [point[0] for point in coordinates]
#         latitudes = [point[1] for point in coordinates]
#         xmin = min(longitudes)
#         xmax = max(longitudes)
#         ymin = min(latitudes)
#         ymax = max(latitudes)

#         # Print and log bounding box information
#         print(f"Processing {name} - xmin: {xmin}, xmax: {xmax}, ymin: {ymin}, ymax: {ymax}")
#         writer.writerow([name, xmin, xmax, ymin, ymax])

# print(f"Bounding box data saved to {output_csv}.")

# Auto2 Download from GEE Asset

In [ ]:
import os
import pandas as pd
from getData import tms_to_geotiff
from PIL import Image
import numpy as np

input_csv = "D:/SIG/Sam2_soucre/Crops/bounding_boxes.csv"
output= "D:/SIG/Sam2_soucre/Crops/imagesZ19/"
os.makedirs(output, exist_ok=True)

df = pd.read_csv(input_csv)
df.head(5)

for index, row in df.iterrows():
    name = row['Area']
    xmin = row['xmin']
    xmax = row['xmax']
    ymin = row['ymin']
    ymax = row['ymax']

    print(f"Processing {name}...")
    print(f"xmin: {xmin}, xmax: {xmax}, ymin: {ymin}, ymax: {ymax}")

    bbox = [xmin, ymin, xmax, ymax]
    output_image = os.path.join(output, f"{name}_z20.tif")

    
    if os.path.exists(output_image):
        print(f"Image for {name} already exists. Skipping download.")
    else:
        satImg = tms_to_geotiff(output=output_image, bbox=bbox, zoom=19, source="Satellite", overwrite=True, return_image=True)
        print(f"Image for {name} saved at {output_image}")

print("All areas processed.")

In [ ]:
import ee
import rasterio
# Initialize Earth Engine
ee.Initialize()

areas_list = [
    "0_vietnam_areas_shp", "10_cambodia_areas_shp", "11_cambodia_areas_shp", "12_cambodia_areas_shp",
    "13_cambodia_areas_shp", "14_cambodia_areas_shp", "15_cambodia_areas_shp", "16_cambodia_areas_shp",
    "17_cambodia_areas_shp", "18_cambodia_areas_shp", "19_cambodia_areas_shp", "1_vietnam_areas_shp",
    "20_cambodia_areas_shp", "21_cambodia_areas_shp", "22_vietnam_areas_shp", "23_vietnam_areas_shp",
    "24_vietnam_areas_shp", "25_vietnam_areas_shp", "26_vietnam_areas_shp", "27_vietnam_areas_shp",
    "28_vietnam_areas_shp", "29_vietnam_areas_shp", "2_cambodia_areas_shp", "30_vietnam_areas_shp",
    "31_vietnam_areas_shp", "32_vietnam_areas_shp", "33_cambodia_areas_shp", "34_cambodia_areas_shp",
    "35_cambodia_areas_shp", "36_cambodia_areas_shp", "37_cambodia_areas_shp", "38_cambodia_areas_shp",
    "39_cambodia_areas_shp", "3_cambodia_areas_shp", "40_vietnam_areas_shp", "41_vietnam_areas_shp",
    "42_vietnam_areas_shp", "43_vietnam_areas_shp", "44_vietnam_areas_shp", "45_vietnam_areas_shp",
    "46_vietnam_areas_shp", "47_vietnam_areas_shp", "48_vietnam_areas_shp", "49_vietnam_areas_shp",
    "4_cambodia_areas_shp", "50_vietnam_areas_shp", "51_vietnam_areas_shp", "52_vietnam_areas_shp",
    "53_vietnam_areas_shp", "54_vietnam_areas_shp", "55_vietnam_areas_shp", "56_vietnam_areas_shp",
    "57_cambodia_areas_shp", "58_cambodia_areas_shp", "59_cambodia_areas_shp", "5_cambodia_areas_shp",
    "60_cambodia_areas_shp", "61_cambodia_areas_shp", "6_cambodia_areas_shp", "7_cambodia_areas_shp",
    "8_cambodia_areas_shp", "9_cambodia_areas_shp"
]

for name in areas_list:
    
    cambodia_areas = ee.FeatureCollection(f'projects/earthengine-legacy/assets/projects/servir-mekong/khProject/data/{name}')
    

    with rasterio.open(f"D:/SIG/Sam2_soucre/Crops/imagesZ19/{name}.tif") as dataset:
        width = dataset.width
        height = dataset.height
    
    print(f"Width: {width}, Height: {height}")

    
    def normalizedDistanceImages(feature):
        
        rasterizedPolygon = ee.Image.constant(0).clip(feature.geometry()).unmask(1, False)
        distance = rasterizedPolygon.distance(kernel=ee.Kernel.euclidean(255), skipMasked=False).clip(feature.geometry()).rename("distance")

        maxDistanceImage = distance.reduceRegion(
            reducer=ee.Reducer.max(),
            geometry=feature.geometry(),
            scale=0.5,
            maxPixels=1e9
        ).get('distance')

        normalizedDistance = distance.divide(ee.Number(maxDistanceImage))
        return normalizedDistance.clip(feature.geometry()).set('system:index', ee.String(feature.get('id')))

    
    normalizedDistanceImages = cambodia_areas.map(normalizedDistanceImages)
    finalNormalizedDistanceImage = ee.ImageCollection(normalizedDistanceImages).mosaic()
    print(f"Band Names: {finalNormalizedDistanceImage.bandNames().getInfo()}")
    dimensions = f"{width}x{height}"

    
    export_task = ee.batch.Export.image.toDrive(
        image=finalNormalizedDistanceImage,
        description=f"Export_Normalized_Distance_{name}",
        folder="FieldZ19",  
        fileNamePrefix=f"{name}_fields",  
        dimensions=dimensions,  
        region=cambodia_areas.geometry().bounds(),  
        scale=None,  
        crs="EPSG:4326",  
        maxPixels=1e12  
    )

    export_task.start()
    print(f"Export task started for {name}!")

# Single Download from GEE Asset

In [ ]:
name = "Sailin493"
# cambodia_areas_10 = ee.FeatureCollection(f'projects/earthengine-legacy/assets/projects/servir-mekong/khProject/data/{name}');
# cambodia_areas_10 = ee.FeatureCollection(f'projects/myanmar-crops/assets/fieldBoundaries/Sailin428');
ft2download = ee.FeatureCollection(f'projects/myanmar-crops/assets/fieldBoundaries/Sailin449');
bounds_info = ft2download.geometry().bounds().getInfo()

# Extract the coordinates from the bounds_info
coordinates = bounds_info['coordinates'][0]

# Calculate xmin, xmax, ymin, ymax
longitudes = [point[0] for point in coordinates]
latitudes = [point[1] for point in coordinates]

xmin = min(longitudes)
xmax = max(longitudes)
ymin = min(latitudes)
ymax = max(latitudes)

print(f'filename: {ft2download}')
print(f"xmin: {xmin}, xmax: {xmax}, ymin: {ymin}, ymax: {ymax}")

In [ ]:
from getData import tms_to_geotiff
from PIL import Image
import numpy as np

bbox = [xmin,
        ymin,
        xmax,
        ymax ]

nr = 1
image = f"D:/SIG/Sam2_soucre/{name}.tif" 
satImg = tms_to_geotiff(output=image, bbox=bbox, zoom=19, source="Satellite", overwrite=True, return_image=True)

In [ ]:
import rasterio
# with rasterio.open('D:/SIG/Sam2_soucre/Crops/source_Z19_2/Images/Sailin493_areas_shp.tif') as dataset:
with rasterio.open(image) as dataset:
    # Get the width (number of columns) and height (number of rows)
    width = dataset.width
    height = dataset.height

print(f"Width: {width}, Height: {height}")

In [ ]:
# cambodia_areas_10

In [ ]:
#// Calculate normalized distance for each polygon separately
#def normalizedDistanceImages = polygons.map(function(feature) {
def normalizedDistanceImages(feature):
  #// Create a rasterized version of the polygon
  #//var rasterizedPolygon = ee.Image.constant(1).clip(feature.geometry())
  rasterizedPolygon = ee.Image.constant(0).clip(feature.geometry()).unmask(1,False);

  #// Calculate the distance from the polygon boundary
  distance = rasterizedPolygon.distance(kernel=ee.Kernel.euclidean(255), skipMasked=False).clip(feature.geometry()).rename("distance"); #// Adjust the distance as needed

  #// Get the max distance as an image by reducing over the polygon geometry
  maxDistanceImage = distance.reduceRegion(
    reducer= ee.Reducer.max(),
    geometry= feature.geometry(),
    scale= 0.5,
    maxPixels= 1e9
  ).get('distance');

  #// Normalize distance within the polygon by dividing by the max distance
  normalizedDistance = distance.divide(ee.Number(maxDistanceImage));

  #// Clip to the polygon and set a unique index for mosaicking
  return normalizedDistance.clip(feature.geometry()).set('system:index', ee.String(feature.get('id')));



#// Merge all normalized distances into a single image
normalizedDistanceImages = cambodia_areas_10.map(normalizedDistanceImages)
finalNormalizedDistanceImage = ee.ImageCollection(normalizedDistanceImages).mosaic();
print(finalNormalizedDistanceImage.bandNames().getInfo())


In [ ]:
# Use the width and height in the dimensions parameter
dimensions = f"{width}x{height}"
# dimensions
# Define export parameters
export_task = ee.batch.Export.image.toDrive(
    image=finalNormalizedDistanceImage,
    description="Export_Normalized_Distance_To_Drive",
    folder="FieldZ19",  # Specify Google Drive folder
    fileNamePrefix=f"{name}_fields",  # File name prefix
    dimensions=dimensions,  # Set dimensions using width and height
    region=cambodia_areas_10.geometry().bounds(),  # Define the region to export
    scale=None,  # Set scale to None since dimensions are specified
    crs="EPSG:4326",  # Coordinate Reference System
    maxPixels=1e12
)

# Start the export task
export_task.start()
print("Export task started!")